# Google の MCP サーバーを使用したエージェントの構築

## 概要

このノートブックでは、Agent Development Kit（ADK）と Model Context Protocol（MCP）サーバーを使用して AI エージェントを構築します。MCP サーバーを活用してエージェントを外部データソースやツール（具体的には Google BigQuery と Google Maps）に接続し、複雑なビジネス上の課題を解決する方法を学びます。<br>
このノートブック全体を通じて、環境をセットアップし、MCP サーバーと直接通信してそのプロトコルを理解し、最終的に ADK 経由でこれらのサーバーを使用するエージェントを構築・テストします。

### シナリオ: ベーカリーの新規開業

これらの概念を実践するために、ロサンゼルスで高級サワードウベーカリーを新規開業しようとしている友人を支援するエージェントを構築します。このエージェントは以下を行う必要があります。
1. BigQuery にクエリを実行して、マクロトレンド（人口動態、競合価格、売上履歴）を把握する。
2. Google Maps を使用して、詳細な立地情報（マイクロロケーション）を検証する。

## 学習目標

このノートブックを完了すると、以下のことができるようになります。
* **Model Context Protocol（MCP）** の概念と、標準化されたツール統合がどのように実現されるかを理解する。
* MCP サーバーと直接通信する。
* MCP サーバーをツールとして使用する ADK エージェントを作成する。
* **ADK Developer UI** を使用して、MCP ツールを含むエージェントの実行フローを操作および検査する。


## セットアップ

このセクションでは、BigQuery に必要なデータをセットアップします。`mcp_bakery` という名前のデータセットを作成し、次の 4 つのテーブルにサンプルデータを読み込みます。
- `demographics`
- `bakery_prices`
- `sales_history_weekly`
- `foot_traffic`

外部スクリプトを実行する代わりに、このノートブック内で直接セットアップ手順を定義します。


In [ ]:
import os
import pandas as pd
import google.auth
from google.cloud import bigquery
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool import StreamableHTTPConnectionParams
import time
import json
import requests
from pprint import pprint


### Google Maps API キーの作成

Google Maps MCP サービス用の制限付き API キーを自動的に作成します。Python を使用して `gcloud` コマンドからの JSON レスポンスを解析し、環境変数 `MAPS_API_KEY` に設定します。


In [ ]:
# gcloud コマンドを実行して出力をキャプチャ
api_key_name = f"bakery-demo-key-{int(time.time())}"
result = !gcloud alpha services api-keys create --display-name="{api_key_name}" --api-target=service=mapstools.googleapis.com --format=json --quiet

result_str = "\n".join(result)

# JSON 以外のテキストをスキップするため最初の { を検索
start = result_str.find("{")

# 見つかった最初の JSON オブジェクトをデコード
decoder = json.JSONDecoder()
key_data, idx = decoder.raw_decode(result_str[start:])

api_key = key_data.get(keyString)
os.environ[MAPS_API_KEY] = api_key


### Google Cloud Storage のデータを使用する

公開バケット `gs://asl-public-data/data/mcp-bakery` に保存されているデータを使用します。次のステップで、このデータを BigQuery に直接読み込みます。


In [ ]:
!gcloud storage ls gs://asl-public-data/data/mcp-bakery/

### BigQuery データセットとテーブルの作成

次に、BigQuery データセットとテーブルを作成し、先ほど確認した CSV ファイルからデータを読み込みます。


In [ ]:
credentials, project_id = google.auth.default()
if not project_id:
    project_id = PROJECT_ID

client = bigquery.Client(project=project_id, credentials=credentials)

dataset_id = f"{project_id}.mcp_bakery"
dataset = bigquery.Dataset(dataset_id)
dataset.location = "US"

try:
    dataset = client.create_dataset(dataset, timeout=30)
    print(f"Created dataset {dataset_id}")
except Exception as e:
    print(f"Dataset might already exist or error: {e}")

# GCS から BQ へ CSV を読み込むヘルパー関数
def load_csv_from_gcs_to_bq(table_name, schema):
    table_id = f"{dataset_id}.{table_name}"
    uri = f"gs://asl-public-data/data/mcp-bakery/{table_name}.csv"
    
    job_config = bigquery.LoadJobConfig(
        schema=schema,
        skip_leading_rows=1,
        source_format=bigquery.SourceFormat.CSV,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    )
    
    job = client.load_table_from_uri(uri, table_id, job_config=job_config)
    job.result()  # ジョブの完了を待機
    print(f"Loaded rows into {table_id} from {uri}.")

# スキーマの定義
demo_schema = [
    bigquery.SchemaField("zip_code", "STRING"),
    bigquery.SchemaField("city", "STRING"),
    bigquery.SchemaField("neighborhood", "STRING"),
    bigquery.SchemaField("total_population", "INT64"),
    bigquery.SchemaField("median_age", "FLOAT64"),
    bigquery.SchemaField("bachelors_degree_pct", "FLOAT64"),
    bigquery.SchemaField("foot_traffic_index", "FLOAT64"),
]

prices_schema = [
    bigquery.SchemaField("store_name", "STRING"),
    bigquery.SchemaField("product_type", "STRING"),
    bigquery.SchemaField("price", "FLOAT64"),
    bigquery.SchemaField("region", "STRING"),
    bigquery.SchemaField("is_organic", "BOOLEAN"),
]

sales_schema = [
    bigquery.SchemaField("week_start_date", "DATE"),
    bigquery.SchemaField("store_location", "STRING"),
    bigquery.SchemaField("product_type", "STRING"),
    bigquery.SchemaField("quantity_sold", "INT64"),
    bigquery.SchemaField("total_revenue", "FLOAT64"),
]

traffic_schema = [
    bigquery.SchemaField("zip_code", "STRING"),
    bigquery.SchemaField("time_of_day", "STRING"),
    bigquery.SchemaField("foot_traffic_score", "FLOAT64"),
]

# テーブルの読み込み
load_csv_from_gcs_to_bq("demographics", demo_schema)
load_csv_from_gcs_to_bq("bakery_prices", prices_schema)
load_csv_from_gcs_to_bq("sales_history_weekly", sales_schema)
load_csv_from_gcs_to_bq("foot_traffic", traffic_schema)


### BigQuery 内のデータの確認

各テーブルの先頭数行を確認して、どのようなデータがあるかを把握し、正しく読み込まれたことを確認します。


In [ ]:
# テーブルの先頭行をクエリして表示する関数
def show_table_head(table_name):
    query = f"SELECT * FROM `{project_id}.mcp_bakery.{table_name}` LIMIT 5"
    df = client.query(query).to_dataframe()
    print(f"\n--- Table: {table_name} ---")
    display(df)

for table in ["demographics", "bakery_prices", "sales_history_weekly", "foot_traffic"]:
    show_table_head(table)


## MCP サーバーの理解

### MCP とは？
Model Context Protocol（MCP）は、AI モデルが外部データソースやツールと安全に対話できるようにするオープンスタンダードです。ツールごとにカスタムの統合コードを作成する代わりに、MCP はモデルが MCP サーバーによって提供される機能を発見し利用するための標準化された方法を提供します。

Google は [多数の Google ホスト型マネージド MCP サーバー](https://docs.cloud.google.com/mcp/supported-products) を提供しています。

このチュートリアルでは、次の 2 つの MCP サーバーを使用します。
1. [**BigQuery MCP**](https://docs.cloud.google.com/bigquery/docs/reference/mcp): エージェントが BigQuery データセットにクエリを実行できるようにします。
2. [**Google Maps MCP**](https://developers.google.com/maps/ai/grounding-lite/reference/mcp): エージェントが Google Maps API（ジオコーディング、プレイスなど）とやり取りできるようにします。


### MCP サーバーの直接呼び出し

MCP が内部でどのように機能するかを理解するために、標準の HTTP リクエストを使用して MCP サーバーを直接呼び出すことができます。MCP サーバーは通常、HTTP または SSE 上の JSON-RPC を使用します。

ここでは、利用可能なツールを一覧表示するために直接 POST リクエストを送信します。


In [ ]:
MAPS_MCP_URL = "https://mapstools.googleapis.com/mcp"
BIGQUERY_MCP_URL = "https://bigquery.googleapis.com/mcp"

ヘッダーをセットアップします。

In [ ]:
# ツールを一覧表示するための BigQuery MCP サーバーへの直接呼び出し
credentials, project_id = google.auth.default(
)
credentials.refresh(google.auth.transport.requests.Request())
oauth_token = credentials.token

headers = {
    "Authorization": f"Bearer {oauth_token}",
    "x-goog-user-project": project_id,
    "Content-Type": "application/json",
}


BigQuery MCP で利用可能なツールを一覧表示してみましょう。

In [ ]:
# MCP のツール一覧リクエストペイロード
payload = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/list",
    "params": {}
}

response = requests.post(BIGQUERY_MCP_URL, headers=headers, json=payload)

resp_json = response.json()
tools = resp_json.get(result, {}).get(tools, [])
print("Available tools:")
for t in tools:
    print(f"- {t.get(name)}")


次に、特定のツールを直接呼び出してみましょう。先ほど作成した `demographics` テーブルにクエリを実行するために、`execute_sql` ツールを呼び出します。出力が長くなりすぎないように、レスポンスのメインコンテンツのみを表示します。

In [ ]:
payload = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/call",
    "params": {
        "name": "execute_sql",
        "arguments": {
            "project_id": project_id,
            "query": f"SELECT * FROM `{project_id}.mcp_bakery.demographics` LIMIT 2"
        }
    }
}

response = requests.post(BIGQUERY_MCP_URL, headers=headers, json=payload)

resp_json = response.json()
result = resp_json.get('result', {})
content = result.get('content', [])

print("Result Content:")
pprint(json.loads(content[0]['text']))


また、Google Maps MCP でどのようなツールが利用可能かも確認してみましょう。

In [ ]:
payload = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/list",
    "params": {}
}

response = requests.post(MAPS_MCP_URL, headers=headers, json=payload)

resp_json = response.json()
tools = resp_json.get('result', {}).get('tools', [])
print("Available tools:")
for t in tools:
    print(f"- {t.get('name')}")

## MCP を使用したエージェントの定義

エージェントとツールセットの定義は、[`mcp_bakery_agents/agent`](./mcp_bakery_agents/agent) 内の専用モジュールに実装されています。
- [tools.py](./mcp_bakery_agents/agent/tools.py): BigQuery と Google Maps の MCP ツールセットを設定します。
- [agent.py](./mcp_bakery_agents/agent/agent.py): ルートエージェントを定義し、ツールセットを Gemini に接続します。

各ファイルの重要な行を確認してみましょう。


### MCP サーバーへの接続（`tools.py`）

[`tools.py`](./mcp_bakery_agents/agent/tools.py) では、ADK の `McpToolset` と `StreamableHTTPConnectionParams` を使用して、リモートの Google Maps および BigQuery MCP サーバーに接続します。

Google Maps MCP サーバーに接続するには、`X-Goog-Api-Key` ヘッダーを介して API キーを渡します。

> ```python
> def get_maps_mcp_toolset():
>     maps_api_key = os.getenv("MAPS_API_KEY", "YOUR_MAPS_API_KEY")
>     return McpToolset(
>         connection_params=StreamableHTTPConnectionParams(
>             url=MAPS_MCP_URL,
>             headers={"X-Goog-Api-Key": maps_api_key},
>         )
>     )
> ```

BigQuery については、Google Cloud アプリケーションのデフォルト認証情報（ADC）を使用して認証し、OAuth トークンとプロジェクトヘッダーを渡します。

> ```python
> def get_bigquery_mcp_toolset():
>     credentials, project_id = google.auth.default(
>         scopes=["https://www.googleapis.com/auth/bigquery"]
>     )
>     credentials.refresh(google.auth.transport.requests.Request())
>     return McpToolset(
>         connection_params=StreamableHTTPConnectionParams(
>             url=BIGQUERY_MCP_URL,
>             headers={
>                 "Authorization": f"Bearer {credentials.token}",
>                 "x-goog-user-project": project_id,
>             },
>         )
>     )
> ```

`McpToolset` は、Server-Sent Events（SSE）を使用して HTTP 経由で MCP エンドポイントに接続します。各サーバーから利用可能なツールを自動的に検出し、Gemini が呼び出し可能なツールへと変換します。


### エージェントの定義（`agent.py`）

[`agent.py`](./mcp_bakery_agents/agent/agent.py) では、MCP ツールセットを初期化し、ADK の `LlmAgent` を使用して `root_agent` を定義します。

> ```python
> bq_toolset = tools.get_bigquery_mcp_toolset()
> maps_toolset = tools.get_maps_mcp_toolset()
>
> root_agent = LlmAgent(
>     model="gemini-3.8-flash",
>     name="bakery_consultant_agent",
>     instruction=PROMPT,
>     tools=[bq_toolset, maps_toolset],
> )
> ```

- **`tools=[bq_toolset, maps_toolset]`**: ADK エージェントは複数のツールセットを受け取ることができます。両方のツールセットを渡すことで、モデルは BigQuery と Google Maps のツールを自由に組み合わせて質問に答えることができます。
- **`root_agent`**: ADK は `adk web` を実行する際、`root_agent` をエントリポイントとして自動的に識別します。


### 環境変数の設定

ADK Developer UI を起動する前に、`mcp_bakery_agents` に `.env` ファイルを設定して、必要な環境変数（Maps API キーや Google Cloud プロジェクト ID など）を提供します。


In [ ]:
os.makedirs('mcp_bakery_agents/agent', exist_ok=True)

with open('mcp_bakery_agents/.env', 'w') as f:
    f.write(f"MAPS_API_KEY={os.environ.get('MAPS_API_KEY', '')}\n")
    f.write(f"GOOGLE_CLOUD_LOCATION=global\n")
    f.write(f"GOOGLE_GENAI_USE_VERTEXAI=TRUE\n")
    f.write(f"PROJECT_ID={project_id}\n")

## ADK Web 経由でのエージェントの呼び出し

`adk web` コマンドを使用して ADK Developer UI を起動し、エージェントとやり取りすることができます。

お使いの環境に応じて、以下の適切なセルを実行してください。


UI が起動したら、エージェントとやり取りできます。次のような質問を試してみてください。

- "Which zip code in Los Angeles has the highest morning foot traffic?"（ロサンゼルスで朝の人出が最も多い郵便番号はどこですか？）
- "Is the density of existing bakeries too high in the Silver Lake area?"（シルバーレイク地域では既存のベーカリーの密度が高すぎますか？）
- "Based on local competitor pricing, what premium price can I charge for a sourdough loaf?"（地元の競合他社の価格設定に基づくと、サワードウブレッド 1 個あたりいくらのプレミアム価格を設定できますか？）
- "Forecast the sales performance based on historical trends."（過去のトレンドに基づいて売上実績を予測してください。）


In [ ]:
# Cloud Workstations の場合
!adk web mcp_bakery_agents --allow_origins "regex:https://.*\.cloudworkstations\.dev"


In [ ]:
# %%bash
# PROXY_BASE=$(curl -s http://metadata.google.internal/computeMetadata/v1/instance/attributes/proxy-url -H "Metadata-Flavor: Google")
# echo "--------------------------------------------------------"
# echo "🔗 ACCESS HERE: https://${PROXY_BASE}/proxy/8000/dev-ui/"
# echo "--------------------------------------------------------"
# adk web mcp_bakery_agents --url_prefix /proxy/8000  --allow_origins "regex:https://.*\.notebooks\.googleusercontent\.com"

## まとめ

このノートブックでは、以下の方法を説明しました。
1. ビジネスシナリオに合わせて BigQuery にサンプルデータをセットアップする。
2. ADK を使用して BigQuery および Google Maps の Google ホスト型 MCP サーバーに接続する。
3. これらの MCP サーバーを活用して複雑なビジネス課題を解決する AI エージェントを定義する。
4. ADK Developer UI を起動してエージェントとやり取りする。


**引用および免責事項**

このノートブックは、公開されているサンプル [Google MCP Launch My Bakery](https://github.com/google/mcp/tree/main/examples/launchmybakery) をベースに更新されたものです。


Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.